# Airbnb Price Prediction — Malaga


In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CITY, OUTPUT_FILE, PROCESSED_DATA_DIR, TARGET_COLUMN
from src.data import load_calendar, load_listings, load_reviews, save_modeling_table

processed_csv = PROCESSED_DATA_DIR / OUTPUT_FILE
print(f"City: {CITY}")

## 1. Load raw data

In [3]:
listings = load_listings()
reviews = load_reviews()
calendar = load_calendar()

print(f"Listings: {listings.shape}")
print(f"Reviews:  {reviews.shape}")
if calendar is not None:
    print(f"Calendar: {calendar.shape}")
else:
    print("Calendar: not downloaded yet (optional)")

listings.head()

Listings: (9714, 79)
Reviews:  (487221, 2)
Calendar: not downloaded yet (optional)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,96033,https://www.airbnb.com/rooms/96033,20250930030808,2025-09-30,city scrape,"Bonito piso a 200m de la playa, El Palo (Málaga)",Do you have a backpacker spirit and are lookin...,"200 metres from the beaches of El Palo, Malaga...",https://a0.muscache.com/pictures/hosting/Hosti...,510467,...,4.93,4.44,4.61,ESFCTU0000290200003588210000000000000000VUT/MA...,f,1,1,0,0,1.88
1,166473,https://www.airbnb.com/rooms/166473,20250930030808,2025-09-30,city scrape,Perfect Location In Malaga,This apartment is rented out by the room - new...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,793360,...,4.91,4.80,4.72,NaN,f,5,1,4,0,0.59
2,330760,https://www.airbnb.com/rooms/330760,20250930030808,2025-09-30,city scrape,Malaga Lodge Guesthouse Double room-shared bath.,The Lodge is set in a charming town house in L...,Málaga Lodge is situated next to the famous Sa...,https://a0.muscache.com/pictures/85419390/38a9...,1687526,...,4.62,4.52,4.48,ESHFTU0000290200004234200060000000000000VFT/MA...,t,6,4,2,0,0.41
3,340024,https://www.airbnb.com/rooms/340024,20250930030808,2025-09-30,city scrape,NEW APARTMENT IN MALAGA CENTER,Welcome to Málaga!<br />This is a modern and e...,It is a central area and has all kinds of serv...,https://a0.muscache.com/pictures/hosting/Hosti...,1725690,...,4.79,4.72,4.77,VFT/MA/02334,f,1,1,0,0,2.11
4,358541,https://www.airbnb.com/rooms/358541,20250930030808,2025-09-30,city scrape,Casa La Maga - Apartment for happy people,"For years, Raúl and I were super happy in this...",The apartment is in the very heart of Malaga C...,https://a0.muscache.com/pictures/miso/Hosting-...,1526932,...,4.97,4.80,4.78,VFT/MA/02288,f,1,1,0,0,2.48


## 2. Cleaning and exploring the data

We clean the raw listings first, save a modeling table, then explore the cleaned data.

### Cleaning and preprocessing

- Convert types (`price`, booleans, dates)
- Drop empty or unused columns
- Remove price outliers and listings outside Malaga bounds
- Aggregate reviews and add `has_reviews`

In [ ]:
from src.data import cleaning_report

processed_csv = save_modeling_table()
df = pd.read_csv(processed_csv)

print(f"Saved {df.shape} -> {processed_csv}")
display(df[TARGET_COLUMN].describe())
display(cleaning_report(df).head(15))

In [ ]:
empty_cols = [c for c in df.columns if df[c].isna().all()]
print("Fully empty columns:", empty_cols if empty_cols else "none")

no_reviews = (df["has_reviews"] == 0).sum()
print(f"Listings without reviews: {no_reviews} ({no_reviews / len(df):.1%})")
print(f"Columns: {df.shape[1]} | Rows: {df.shape[0]}")

### Price distribution

Nightly price on the cleaned dataset (histogram up to 500 EUR).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, OUTPUT_FILE, PROCESSED_DATA_DIR, TARGET_COLUMN

df = pd.read_csv(processed_csv)
prices = df[TARGET_COLUMN]

print(f"Listings: {len(prices)}")
print(f"Median: {prices.median():.0f} EUR | Mean: {prices.mean():.0f} EUR")
print(f"Max: {prices.max():.0f} EUR | Above 500 EUR: {(prices > 500).sum()}")

plt.figure(figsize=(8, 4))
plt.hist(prices[prices <= 500], bins=30, edgecolor="white", color="steelblue")
plt.axvline(prices.median(), color="red", linestyle="--", label=f"Median = {prices.median():.0f} EUR")
plt.xlabel("Price per night (EUR)")
plt.ylabel("Number of listings")
plt.title("Price distribution (up to 500 EUR)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_price_distribution.png", dpi=150)
plt.show()

Most listings fall between 75 and 150 EUR. A small number of expensive listings creates a right-skewed distribution.

### Missing values

Columns with missing data after cleaning, and the main reasons.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, OUTPUT_FILE, PROCESSED_DATA_DIR
from src.data import cleaning_report

df = pd.read_csv(processed_csv)
report = cleaning_report(df)
top_missing = report[report["missing_pct"] > 0].head(15)

display(top_missing)

plt.figure(figsize=(8, 5))
plt.barh(top_missing["column"], top_missing["missing_pct"], color="coral")
plt.xlabel("Missing (%)")
plt.title("Columns with the most missing values")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_missing_values.png", dpi=150)
plt.show()

no_reviews = df["has_reviews"] == 0
print(f"Listings without reviews: {no_reviews.sum()} ({no_reviews.mean():.1%})")
print("Review scores are missing when a listing has no reviews.")

## 3. Baseline model (tabular + spatial)

Linear regression on tabular features and coordinates. Train/test split: 80% / 20%. Metrics: MAE, RMSE, R².

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import OUTPUT_FILE, PROCESSED_DATA_DIR, RANDOM_SEED, TARGET_COLUMN, TEST_SIZE
from src.modeling import (
    build_tabular_preprocessor,
    get_categorical_columns,
    get_numeric_columns,
    print_metrics,
    regression_metrics,
)

df = pd.read_csv(processed_csv)

exclude = [
    "id", TARGET_COLUMN, "review_text",
    "name", "description", "neighborhood_overview", "host_about",
    "amenities", "bathrooms_text",
    "first_review", "last_review", "first_review_date", "last_review_date", "host_since",
]

numeric_features = get_numeric_columns(df, exclude=exclude)
categorical_features = get_categorical_columns(df, exclude=exclude)

X = df[numeric_features + categorical_features]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED,
)

prep = build_tabular_preprocessor(numeric_features, categorical_features)
lr_pipe = Pipeline([("prep", prep), ("model", LinearRegression())])
lr_pipe.fit(X_train, y_train)

y_pred = lr_pipe.predict(X_test)
metrics = regression_metrics(y_test, y_pred)
print_metrics(metrics, label="Linear Regression")


### 3.1 Model results

Predicted vs true prices, residuals, and coefficient distribution.

In [ ]:
from src.config import FIGURES_DIR
from src.modeling import plot_regression_results

weights = lr_pipe.named_steps["model"].coef_
plot_regression_results(
    y_test,
    y_pred,
    weights,
    target_label="price (EUR)",
    save_path=FIGURES_DIR / "03_linear_regression_results.png",
)


## 4. Hybrid model (text + tabular)

TF-IDF on descriptions and reviews combined with tabular features.

## 5. Model comparison

Compare models and save results to `reports/figures/`.